# FASE 1: Ingesta, Conversión y Consolidación de Tablas Master NHANES (2017-2018)
Este notebook ha sido refactorizado para utilizar el **Data Catalog** de Kedro en lugar de rutas hardcodeadas. La inicialización de la extensión de Kedro provee acceso global al objeto `catalog`.

In [ ]:
# Inicializar la sesión de Kedro en el notebook
%load_ext kedro.ipython

## Paso 1.1: Conversión de Formato (XPT a Parquet)
Utilizamos `catalog.load()` para cargar los datos en crudo (`_raw`) directamente como DataFrames, estandarizamos sus columnas, y utilizamos `catalog.save()` para persistirlos en formato intermedio (`_intermediate`).

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def convert_raw_to_intermediate(dataset_name: str):
    """Carga un dataset raw del catálogo, lo estandariza y lo guarda como intermediate."""
    try:
        # Carga delegada a Kedro
        df = catalog.load(f"{dataset_name}_raw")
        
        # Transformación
        df.columns = df.columns.str.upper()
        
        # Guardado delegado a Kedro
        catalog.save(f"{dataset_name}_intermediate", df)
        print(f"✅ Procesado y guardado en catálogo: {dataset_name}_intermediate | Shape: {df.shape}")
        
    except Exception as e:
        print(f"❌ Error al procesar {dataset_name}: {e}")

# Ejecución de ejemplo para Demographics
# convert_raw_to_intermediate('demo_j')

## Paso 1.2: Consolidación de Tablas Master por Sección
Agrupación utilizando únicamente accesos al catálogo (`catalog.load` de los intermedios y `catalog.save` a los masters).

In [ ]:
def build_master_table(section_name: str, dataset_names: list):
    """Une múltiples datasets intermedios usando SEQN y guarda el master en el catálogo."""
    print(f"\n--- Construyendo Master para sección: {section_name.upper()} ---")
    master_df = None
    
    for ds_name in dataset_names:
        try:
            df = catalog.load(f"{ds_name}_intermediate")
            
            if 'SEQN' not in df.columns:
                print(f"⚠️ Columna clave 'SEQN' no encontrada en {ds_name}. Saltando...")
                continue
                
            if master_df is None:
                master_df = df
                print(f"🟢 Base inicial ({ds_name}): {master_df.shape}")
            else:
                initial_rows = master_df.shape[0]
                master_df = master_df.merge(df, on='SEQN', how='outer')
                print(f"🔄 Join con {ds_name}: {df.shape} -> Master actual: {master_df.shape}")
                
                if master_df.shape[0] < initial_rows:
                    print(f"⚠️ ALERTA: Posible pérdida de filas.")
                    
        except Exception as e:
            print(f"❌ Error al procesar {ds_name}: {e}")

    if master_df is not None:
        master_name = f"{section_name}_master"
        catalog.save(master_name, master_df)
        print(f"✅ MASTER CREADO Y GUARDADO EN CATÁLOGO: {master_name} | Shape Final: {master_df.shape}")
    else:
        print(f"❌ No se pudo crear el master para {section_name}")

# Ejecución de ejemplo:
# build_master_table('demographics', ['demo_j'])